In [ ]:
from gdrive_fsspec import GoogleDriveFileSystem
import marimo as mo

fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})
mo.output.clear_console()

In [ ]:
fs.get("Data_Science_Project/data/cache_deleaked.pt", "/marimo/cache_deleaked.pt")

In [ ]:
#!/usr/bin/env python3
"""
Train MLP (3‑layer) with triplet loss on extended cache.
"""

import os
import math
import random
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from tqdm import tqdm

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
class Config:
    CACHE_PATH = "/marimo/cache_deleaked.pt"            # point to your extended cache
    CHECKPOINT_DIR = "./checkpoints"
    CHECKPOINT_PATH = "./checkpoints/mlp.pt"
    FINAL_CHECKPOINT_PATH = "./checkpoints/mlp_final.pt"

    EMBEDDING_DIM = 192
    DROPOUT = 0.15

    BATCH_SIZE = 4096
    EVAL_BATCH_SIZE = 8192
    EPOCHS = 150
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    GRAD_CLIP = 1.0
    USE_AMP = True
    COMPILE_MODEL = True
    SAVE_EVERY_N_BATCHES = 1000

    TRIPLET_MARGIN = 0.25
    K_NEGATIVES = 16

    SCHEDULER_FACTOR = 0.5
    SCHEDULER_PATIENCE = 5
    EARLY_STOPPING_PATIENCE = 35

    NUM_EVAL_PAIRS = 10000
    EVAL_SEED = 123

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)


# ----------------------------------------------------------------------
# MODEL
# ----------------------------------------------------------------------
class MLP(nn.Module):
    def __init__(self, embedding_dim=192, dropout=0.15):
        super().__init__()
        self.fc1 = nn.Linear(2 * embedding_dim, embedding_dim)
        self.fc2 = nn.Linear(embedding_dim, embedding_dim)
        self.fc3 = nn.Linear(embedding_dim, embedding_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, noisy_emb, enhanced_emb):
        x = torch.cat([noisy_emb, enhanced_emb], dim=-1)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        fused = self.fc3(x)
        return F.normalize(fused, p=2, dim=-1)


# ----------------------------------------------------------------------
# EVALUATION (anchor‑vs‑fused)
# ----------------------------------------------------------------------
def build_speaker_index(meta):
    speaker_to_indices = defaultdict(list)
    for idx, row in meta.iterrows():
        key = (row["language"], row["speaker"])
        speaker_to_indices[key].append(int(idx))
    return dict(speaker_to_indices)

def compute_eer(gen_scores, imp_scores):
    scores = np.concatenate([gen_scores, imp_scores])
    labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
    order = np.argsort(scores)
    scores, labels = scores[order], labels[order]
    n_gen = np.sum(labels == 1)
    n_imp = np.sum(labels == 0)
    fnr = np.cumsum(labels == 1) / n_gen if n_gen > 0 else 0
    fpr = 1.0 - np.cumsum(labels == 0) / n_imp if n_imp > 0 else 0
    idx = np.argmin(np.abs(fnr - fpr))
    return (fnr[idx] + fpr[idx]) / 2.0 * 100.0

@torch.no_grad()
def evaluate_anchor_vs_fused(model, cache, device, batch_size=8192):
    model.eval()
    meta = cache["meta"].reset_index(drop=True)
    clean_anchor = cache["clean_anchor"]
    noisy = torch.from_numpy(cache["noisy_emb"]).float()
    enhanced = torch.from_numpy(cache["enhanced_emb"]).float()

    fused_chunks = []
    for start in range(0, len(noisy), batch_size):
        end = min(start + batch_size, len(noisy))
        out = model(noisy[start:end].to(device), enhanced[start:end].to(device))
        fused = out[0] if isinstance(out, tuple) else out
        fused_chunks.append(F.normalize(fused, p=2, dim=-1).cpu())
    fused = torch.cat(fused_chunks, dim=0)

    speaker_to_anchor = {}
    for key, anchor in clean_anchor.items():
        if not isinstance(anchor, torch.Tensor):
            anchor = torch.as_tensor(anchor, dtype=torch.float32)
        speaker_to_anchor[key] = anchor

    rng = np.random.RandomState(123)
    speaker_list = list(speaker_to_anchor.keys())
    gen_scores, imp_scores = [], []

    for idx in range(len(fused)):
        row = meta.iloc[idx]
        spk_key = (row["language"], row["speaker"])
        if spk_key not in speaker_to_anchor:
            continue
        anchor = speaker_to_anchor[spk_key].to(device)
        emb = fused[idx].unsqueeze(0).to(device)

        gen = F.cosine_similarity(emb, anchor.unsqueeze(0), dim=1).item()
        gen_scores.append(gen)

        other_indices = [i for i, s in enumerate(speaker_list) if s != spk_key]
        if other_indices:
            other_idx = rng.choice(other_indices)
            other = speaker_list[other_idx]
            other_anchor = speaker_to_anchor[other].to(device)
            imp = F.cosine_similarity(emb, other_anchor.unsqueeze(0), dim=1).item()
            imp_scores.append(imp)

    eer = compute_eer(gen_scores, imp_scores)
    return {"eer": eer, "genuine_mean": np.mean(gen_scores), "impostor_mean": np.mean(imp_scores)}


# ----------------------------------------------------------------------
# LOAD CACHE
# ----------------------------------------------------------------------
def load_cache_safe(cache_path):
    print(f"Loading cache from {cache_path}")
    data = torch.load(cache_path, map_location="cpu", weights_only=False)
    if not isinstance(data, dict):
        raise TypeError(f"Unexpected cache type: {type(data)}")
    if "meta" in data:
        meta_data = data["meta"]
        if isinstance(meta_data, pd.DataFrame):
            meta = meta_data.copy()
        elif isinstance(meta_data, list):
            meta = pd.DataFrame(meta_data)
        else:
            raise TypeError(f"Unexpected meta type: {type(meta_data)}")
    elif "meta_records" in data:
        meta = pd.DataFrame(data["meta_records"])
    else:
        raise KeyError("Cache missing 'meta' or 'meta_records'.")
    noisy_emb = data.get("noisy_emb")
    enhanced_emb = data.get("enhanced_emb")
    if noisy_emb is None or enhanced_emb is None:
        raise ValueError("Cache missing noisy_emb or enhanced_emb.")
    if torch.is_tensor(noisy_emb):
        noisy_emb = noisy_emb.cpu().numpy()
    else:
        noisy_emb = np.asarray(noisy_emb)
    if torch.is_tensor(enhanced_emb):
        enhanced_emb = enhanced_emb.cpu().numpy()
    else:
        enhanced_emb = np.asarray(enhanced_emb)
    meta = meta.reset_index(drop=True)
    return {"meta": meta, "noisy_emb": noisy_emb.astype(np.float32),
            "enhanced_emb": enhanced_emb.astype(np.float32),
            "clean_anchor": data.get("clean_anchor")}


# ----------------------------------------------------------------------
# TRAIN
# ----------------------------------------------------------------------
def train():
    device = torch.device(Config.DEVICE)
    print(f"Device: {device}")

    cache = load_cache_safe(Config.CACHE_PATH)
    meta = cache["meta"]

    # Speaker mapping
    clean_anchor = cache["clean_anchor"]
    valid_spk_keys = list(clean_anchor.keys())
    spk_to_idx = {k: i for i, k in enumerate(valid_spk_keys)}
    anchor_stack = torch.stack([
        torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in valid_spk_keys
    ]).to(device, non_blocking=True)
    anchor_stack = F.normalize(anchor_stack, p=2, dim=1)

    lang_arr = meta["language"].to_numpy()
    spk_arr = meta["speaker"].to_numpy()
    row_anchor_idx = np.array(
        [spk_to_idx.get((l, s), -1) for l, s in zip(lang_arr, spk_arr)],
        dtype=np.int64
    )
    has_anchor = row_anchor_idx >= 0
    row_anchor_idx_t = torch.from_numpy(np.maximum(row_anchor_idx, 0)).long().to(device, non_blocking=True)

    train_idx = meta.index[(meta["split"] == "train") & has_anchor].to_numpy(dtype=np.int64)
    val_idx = meta.index[(meta["split"] == "val") & has_anchor].to_numpy(dtype=np.int64)
    test_idx = meta.index[(meta["split"] == "test") & has_anchor].to_numpy(dtype=np.int64)

    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

    val_cache = {
        "meta": meta.iloc[val_idx].copy().reset_index(drop=True),
        "noisy_emb": cache["noisy_emb"][val_idx],
        "enhanced_emb": cache["enhanced_emb"][val_idx],
        "clean_anchor": clean_anchor,
    }

    # Pre-load embeddings
    noisy_t = torch.from_numpy(cache["noisy_emb"]).float().to(device, non_blocking=True)
    enh_t = torch.from_numpy(cache["enhanced_emb"]).float().to(device, non_blocking=True)

    # Speaker groups for negative sampling
    by_speaker = meta.groupby(["language", "speaker"]).groups
    spk_rows = {k: np.array(list(v)) for k, v in by_speaker.items()}
    lang_speakers = defaultdict(list)
    for (lang, spk) in spk_rows:
        lang_speakers[lang].append(spk)
    other_speakers_map = {}
    for lang, spks in lang_speakers.items():
        spks_arr = np.array(spks, dtype=object)
        for spk in spks:
            other_speakers_map[(lang, spk)] = spks_arr[spks_arr != spk]

    # Model
    model = MLP(embedding_dim=Config.EMBEDDING_DIM, dropout=Config.DROPOUT).to(device)
    if Config.COMPILE_MODEL and hasattr(torch, 'compile'):
        print("Compiling model...")
        model = torch.compile(model, mode='reduce-overhead', dynamic=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                            factor=Config.SCHEDULER_FACTOR,
                                                            patience=Config.SCHEDULER_PATIENCE)
    autocast_enabled = Config.USE_AMP and torch.cuda.is_available()
    scaler = GradScaler(device="cuda") if autocast_enabled else None

    config_dict = {
        k: getattr(Config, k) for k in dir(Config)
        if not k.startswith('_') and not callable(getattr(Config, k))
    }

    start_epoch = 0
    best_val_eer = float('inf')
    best_state = None
    no_improve = 0

    # Resume logic (simplified)
    if os.path.exists(Config.CHECKPOINT_PATH):
        ckpt = torch.load(Config.CHECKPOINT_PATH, map_location=device, weights_only=False)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch = ckpt["epoch"] + 1
        best_val_eer = ckpt["best_val_eer"]
        best_state = ckpt.get("best_state")
        no_improve = ckpt.get("no_improve", 0)
        print(f"Resuming from epoch {start_epoch}, best EER {best_val_eer:.2f}%")

    K = Config.K_NEGATIVES
    margin = Config.TRIPLET_MARGIN
    batch_size = Config.BATCH_SIZE

    print(f"Triplet margin: {margin}, K negatives: {K}")

    for epoch in range(start_epoch, Config.EPOCHS):
        model.train()
        rng = np.random.RandomState(42 + epoch)
        perm = rng.permutation(train_idx)

        epoch_losses = []
        batch_count = 0

        pbar = tqdm(range(0, len(perm), batch_size), desc=f"Epoch {epoch+1}/{Config.EPOCHS}")
        for start in pbar:
            batch_rows = perm[start:start + batch_size]
            if len(batch_rows) < 2:
                continue

            optimizer.zero_grad(set_to_none=True)

            b_noisy = noisy_t[batch_rows]
            b_enh = enh_t[batch_rows]
            b_anchor = anchor_stack[row_anchor_idx_t[batch_rows]]

            with autocast(device_type="cuda", enabled=autocast_enabled):
                pos_fused = model(b_noisy, b_enh)

                # Sample K negatives
                neg_rows = np.empty((len(batch_rows), K), dtype=np.int64)
                for bi, r in enumerate(batch_rows):
                    lang = lang_arr[r]
                    spk = spk_arr[r]
                    others = other_speakers_map[(lang, spk)]
                    for k_idx in range(K):
                        neg_spk = others[rng.randint(len(others))]
                        cand = spk_rows[(lang, neg_spk)]
                        neg_rows[bi, k_idx] = cand[rng.randint(len(cand))]

                neg_rows_flat = neg_rows.reshape(-1)
                neg_noisy = noisy_t[neg_rows_flat]
                neg_enh = enh_t[neg_rows_flat]

                with torch.no_grad():
                    neg_fused_flat = model(neg_noisy, neg_enh)
                    neg_fused = neg_fused_flat.view(len(batch_rows), K, -1)

                    d_pos = 1 - F.cosine_similarity(b_anchor, pos_fused.detach(), dim=-1)
                    d_negs = 1 - F.cosine_similarity(
                        b_anchor.unsqueeze(1).expand(-1, K, -1),
                        neg_fused, dim=-1
                    )
                    semi_hard_mask = (d_negs > d_pos.unsqueeze(1)) & (d_negs < (d_pos + margin).unsqueeze(1))
                    masked_d = torch.where(semi_hard_mask, d_negs, torch.full_like(d_negs, float("inf")))
                    has_semi_hard = semi_hard_mask.any(dim=1)
                    fallback_idx = d_negs.argmin(dim=1)
                    semi_idx = torch.where(has_semi_hard, masked_d.argmin(dim=1), fallback_idx)

                semi_idx_np = semi_idx.detach().cpu().numpy()
                sel_neg_rows = neg_rows[np.arange(len(batch_rows)), semi_idx_np]
                sel_neg_fused = model(noisy_t[sel_neg_rows], enh_t[sel_neg_rows])

                d_pos_active = 1 - F.cosine_similarity(b_anchor, pos_fused, dim=-1)
                d_neg_active = 1 - F.cosine_similarity(b_anchor, sel_neg_fused, dim=-1)
                loss = F.relu(d_pos_active - d_neg_active + margin).mean()

            if not torch.isfinite(loss):
                continue

            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
                optimizer.step()

            epoch_losses.append(loss.item())
            batch_count += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

            if Config.SAVE_EVERY_N_BATCHES > 0 and batch_count % Config.SAVE_EVERY_N_BATCHES == 0:
                torch.save({
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "scaler_state": scaler.state_dict() if scaler else None,
                    "best_val_eer": best_val_eer,
                    "best_state": best_state,
                    "no_improve": no_improve,
                    "config": config_dict,
                }, Config.CHECKPOINT_PATH + ".intermediate")
                print(f"\n[Intermediate] Saved at batch {batch_count}")

        # Validation
        val_result = evaluate_anchor_vs_fused(model, val_cache, device, batch_size=Config.EVAL_BATCH_SIZE)
        val_eer = val_result["eer"]
        scheduler.step(val_eer)

        print(f"\nEpoch {epoch+1:03d} | Loss: {np.mean(epoch_losses):.5f} | Val EER: {val_eer:.2f}%")
        print(f"  Genuine mean: {val_result['genuine_mean']:.4f}, Impostor mean: {val_result['impostor_mean']:.4f}")

        if val_eer < best_val_eer:
            best_val_eer = val_eer
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            print(f"  *** NEW BEST: {best_val_eer:.2f}% ***")
        else:
            no_improve += 1

        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler_state": scaler.state_dict() if scaler else None,
            "best_val_eer": best_val_eer,
            "best_state": best_state,
            "no_improve": no_improve,
            "config": config_dict,
        }, Config.CHECKPOINT_PATH)

        if no_improve >= Config.EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    # Restore best
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final test
    test_cache = {
        "meta": meta.iloc[test_idx].copy().reset_index(drop=True),
        "noisy_emb": cache["noisy_emb"][test_idx],
        "enhanced_emb": cache["enhanced_emb"][test_idx],
        "clean_anchor": clean_anchor,
    }
    test_result = evaluate_anchor_vs_fused(model, test_cache, device, batch_size=Config.EVAL_BATCH_SIZE)

    print("\n" + "="*70)
    print("FINAL TEST RESULTS (anchor‑vs‑fused)")
    print("="*70)
    print(f"EER: {test_result['eer']:.2f}%")
    print(f"Genuine mean: {test_result['genuine_mean']:.4f}")
    print(f"Impostor mean: {test_result['impostor_mean']:.4f}")

    torch.save({
        "gate_state": model.state_dict(),
        "model_type": "mlp",
        "embedding_dim": Config.EMBEDDING_DIM,
        "test_results": test_result,
    }, Config.FINAL_CHECKPOINT_PATH)
    print(f"\nSaved final checkpoint to {Config.FINAL_CHECKPOINT_PATH}")

if __name__ == "__main__":
    train()

In [ ]:
def self():
    #!/usr/bin/env python3
    """
    Train Self-Attention (over [noisy, enhanced]) with triplet loss on extended cache.
    Optimized for RTX Ada/Ampere architectures (TF32/FP16, Vectorized Ops).
    """

    import os
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.amp import autocast
    from collections import defaultdict
    from tqdm import tqdm

    # Enable TF32 for massive speedups on Ampere/Ada
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # ----------------------------------------------------------------------
    # KERNEL FIX: Disable Flash Attention
    # Reason 1: Seq length is 2. Flash attention introduces launch overhead.
    # Reason 2: Head dim is 48 (192/4). Flash attention natively prefers 32/64/128.
    # ----------------------------------------------------------------------
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)


    # ----------------------------------------------------------------------
    # CONFIG
    # ----------------------------------------------------------------------
    class Config:
        CACHE_PATH = "./cache_deleaked.pt"
        CHECKPOINT_DIR = "./checkpoints"
        CHECKPOINT_PATH = "./checkpoints/self_attention.pt"
        FINAL_CHECKPOINT_PATH = "./checkpoints/self_attention_final.pt"

        EMBEDDING_DIM = 192
        NUM_HEADS = 4
        DROPOUT = 0.1

        BATCH_SIZE = 4096
        EVAL_BATCH_SIZE = 16384
        EPOCHS = 150
        LR = 1e-3
        WEIGHT_DECAY = 1e-4
        GRAD_CLIP = 1.0
        USE_AMP = True
        COMPILE_MODEL = True          # <-- Restored: Safe to compile now
        SAVE_EVERY_N_BATCHES = 1000

        TRIPLET_MARGIN = 0.25
        K_NEGATIVES = 16

        SCHEDULER_FACTOR = 0.5
        SCHEDULER_PATIENCE = 5
        EARLY_STOPPING_PATIENCE = 35

        EVAL_SEED = 123
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)


    # ----------------------------------------------------------------------
    # MODEL
    # ----------------------------------------------------------------------
    class SelfAttentionFusion(nn.Module):
        def __init__(self, embedding_dim=192, num_heads=4, dropout=0.1):
            super().__init__()
            assert embedding_dim % num_heads == 0
            self.proj = nn.Linear(embedding_dim, embedding_dim)
            self.attn = nn.MultiheadAttention(
                embed_dim=embedding_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
            self.norm1 = nn.LayerNorm(embedding_dim)
            self.norm2 = nn.LayerNorm(embedding_dim)
            self.mlp = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(embedding_dim * 2, embedding_dim),
                nn.Dropout(dropout),
            )
            self.out_proj = nn.Linear(embedding_dim, embedding_dim)

        def forward(self, noisy_emb, enhanced_emb):
            n = self.proj(noisy_emb)
            e = self.proj(enhanced_emb)
    
            # FIX: Ensure contiguous memory layout for the attention backend
            x = torch.stack([n, e], dim=1).contiguous() 
    
            attn_out, _ = self.attn(x, x, x, need_weights=False)
            x = self.norm1(x + attn_out)
            pooled = x.mean(dim=1)
            mlp_out = self.mlp(pooled)
            out = self.norm2(pooled + mlp_out)
            fused = self.out_proj(out)
            return F.normalize(fused, p=2, dim=-1)


    # ----------------------------------------------------------------------
    # EVALUATION
    # ----------------------------------------------------------------------
    def compute_eer(gen_scores, imp_scores):
        scores = np.concatenate([gen_scores, imp_scores])
        labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
        order = np.argsort(scores)
        scores, labels = scores[order], labels[order]
        n_gen = np.sum(labels == 1)
        n_imp = np.sum(labels == 0)
        fnr = np.cumsum(labels == 1) / n_gen if n_gen > 0 else 0
        fpr = 1.0 - np.cumsum(labels == 0) / n_imp if n_imp > 0 else 0
        idx = np.argmin(np.abs(fnr - fpr))
        return (fnr[idx] + fpr[idx]) / 2.0 * 100.0


    @torch.no_grad()
    def evaluate_anchor_vs_fused(model, cache, device, batch_size=16384):
        model.eval()
        meta = cache["meta"].reset_index(drop=True)
        clean_anchor = cache["clean_anchor"]
        noisy = torch.from_numpy(cache["noisy_emb"]).float().to(device)
        enhanced = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

        fused_chunks = []
        for start in range(0, len(noisy), batch_size):
            end = min(start + batch_size, len(noisy))
            with autocast(device_type="cuda", dtype=torch.float16, enabled=Config.USE_AMP):
                fused_chunks.append(model(noisy[start:end], enhanced[start:end]).float())
        fused = torch.cat(fused_chunks, dim=0)

        spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(spk_keys)}
        anchor_matrix = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in spk_keys
        ]).to(device)
        anchor_matrix = F.normalize(anchor_matrix, p=2, dim=1)

        sample_spk_ids = np.array([spk_to_idx.get((r["language"], r["speaker"]), -1) for _, r in meta.iterrows()])
        valid_mask = sample_spk_ids >= 0
        valid_fused = fused[valid_mask]
        valid_ids = torch.from_numpy(sample_spk_ids[valid_mask]).to(device)

        gen_scores = F.cosine_similarity(valid_fused, anchor_matrix[valid_ids], dim=1).cpu().numpy()

        rng = np.random.RandomState(Config.EVAL_SEED)
        num_spks = len(spk_keys)
        shift = rng.randint(1, num_spks, size=len(valid_ids))
        imp_ids = torch.from_numpy((sample_spk_ids[valid_mask] + shift) % num_spks).to(device)
        imp_scores = F.cosine_similarity(valid_fused, anchor_matrix[imp_ids], dim=1).cpu().numpy()

        eer = compute_eer(gen_scores, imp_scores)
        return {"eer": eer, "genuine_mean": float(np.mean(gen_scores)), "impostor_mean": float(np.mean(imp_scores))}


    # ----------------------------------------------------------------------
    # LOAD CACHE
    # ----------------------------------------------------------------------
    def load_cache_safe(cache_path):
        print(f"Loading cache from {cache_path}")
        import zipfile
        import pickle
        try:
            data = torch.load(cache_path, map_location="cpu", weights_only=False)
        except TypeError as e:
            if "StringDtype" in str(e):
                print("Pandas version mismatch – falling back to manual unpickle...")
                with zipfile.ZipFile(cache_path, 'r') as zf:
                    with zf.open('data.pkl') as f:
                        data = pickle.load(f)
            else:
                raise

        if "meta" in data:
            meta_data = data["meta"]
            if isinstance(meta_data, pd.DataFrame):
                meta = meta_data.copy()
            elif isinstance(meta_data, list):
                meta = pd.DataFrame(meta_data)
            else:
                raise TypeError(f"Unexpected meta type: {type(meta_data)}")
        elif "meta_records" in data:
            meta = pd.DataFrame(data["meta_records"])
        else:
            raise KeyError("Cache missing 'meta' or 'meta_records'.")

        noisy_emb = data["noisy_emb"].cpu().numpy() if torch.is_tensor(data["noisy_emb"]) else np.asarray(data["noisy_emb"])
        enhanced_emb = data["enhanced_emb"].cpu().numpy() if torch.is_tensor(data["enhanced_emb"]) else np.asarray(data["enhanced_emb"])

        return {
            "meta": meta.reset_index(drop=True),
            "noisy_emb": noisy_emb.astype(np.float32),
            "enhanced_emb": enhanced_emb.astype(np.float32),
            "clean_anchor": data["clean_anchor"]
        }


    # ----------------------------------------------------------------------
    # TRAIN
    # ----------------------------------------------------------------------
    def train():
        device = torch.device(Config.DEVICE)
        print(f"Device: {device}")

        cache = load_cache_safe(Config.CACHE_PATH)
        meta = cache["meta"]

        clean_anchor = cache["clean_anchor"]
        valid_spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(valid_spk_keys)}
        anchor_stack = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in valid_spk_keys
        ]).to(device, non_blocking=True)
        anchor_stack = F.normalize(anchor_stack, p=2, dim=1)

        lang_arr = meta["language"].to_numpy()
        spk_arr = meta["speaker"].to_numpy()
        row_anchor_idx = np.array([spk_to_idx.get((l, s), -1) for l, s in zip(lang_arr, spk_arr)], dtype=np.int64)

        has_anchor = row_anchor_idx >= 0
        row_anchor_idx_t = torch.from_numpy(np.maximum(row_anchor_idx, 0)).long().to(device, non_blocking=True)

        train_idx = meta.index[(meta["split"] == "train") & has_anchor].to_numpy(dtype=np.int64)
        val_idx = meta.index[(meta["split"] == "val") & has_anchor].to_numpy(dtype=np.int64)
        test_idx = meta.index[(meta["split"] == "test") & has_anchor].to_numpy(dtype=np.int64)

        print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

        val_cache = {
            "meta": meta.iloc[val_idx].copy().reset_index(drop=True),
            "noisy_emb": cache["noisy_emb"][val_idx],
            "enhanced_emb": cache["enhanced_emb"][val_idx],
            "clean_anchor": clean_anchor,
        }

        noisy_t = torch.from_numpy(cache["noisy_emb"]).float().to(device, non_blocking=True)
        enh_t = torch.from_numpy(cache["enhanced_emb"]).float().to(device, non_blocking=True)

        by_speaker = meta.groupby(["language", "speaker"]).groups
        spk_rows = {k: np.array(list(v)) for k, v in by_speaker.items()}
        lang_speakers = defaultdict(list)
        for (lang, spk) in spk_rows:
            lang_speakers[lang].append(spk)

        other_speakers_map = {}
        for lang, spks in lang_speakers.items():
            spks_arr = np.array(spks, dtype=object)
            for spk in spks:
                other_speakers_map[(lang, spk)] = spks_arr[spks_arr != spk]

        model = SelfAttentionFusion(
            embedding_dim=Config.EMBEDDING_DIM,
            num_heads=Config.NUM_HEADS,
            dropout=Config.DROPOUT,
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=Config.SCHEDULER_FACTOR, patience=Config.SCHEDULER_PATIENCE)

        start_epoch = 0
        best_val_eer = float('inf')
        best_state = None
        no_improve = 0

        if os.path.exists(Config.CHECKPOINT_PATH):
            ckpt = torch.load(Config.CHECKPOINT_PATH, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scheduler.load_state_dict(ckpt["scheduler"])
            start_epoch = ckpt["epoch"] + 1
            best_val_eer = ckpt["best_val_eer"]
            best_state = ckpt.get("best_state")
            no_improve = ckpt.get("no_improve", 0)
            print(f"Resuming from epoch {start_epoch}, best EER {best_val_eer:.2f}%")

        if Config.COMPILE_MODEL and hasattr(torch, 'compile'):
            print("Compiling model for maximum speed...")
            model = torch.compile(model)   
    
        K = Config.K_NEGATIVES
        margin = Config.TRIPLET_MARGIN
        batch_size = Config.BATCH_SIZE

        config_dict = {k: getattr(Config, k) for k in dir(Config) if not k.startswith('_') and not callable(getattr(Config, k))}

        for epoch in range(start_epoch, Config.EPOCHS):
            model.train()
            rng = np.random.RandomState(42 + epoch)
            perm = rng.permutation(train_idx)

            all_neg_rows = np.zeros((len(perm), K), dtype=np.int64)
            for i, r in enumerate(perm):
                lang, spk = lang_arr[r], spk_arr[r]
                others = other_speakers_map[(lang, spk)]
                rand_spks = others[rng.randint(0, len(others), size=K)]
                for k_idx, neg_spk in enumerate(rand_spks):
                    cand = spk_rows[(lang, neg_spk)]
                    all_neg_rows[i, k_idx] = cand[rng.randint(len(cand))]

            epoch_losses = []
            batch_count = 0
            pbar = tqdm(range(0, len(perm), batch_size), desc=f"Epoch {epoch+1}/{Config.EPOCHS}")

            for start in pbar:
                batch_rows = perm[start:start + batch_size]
                if len(batch_rows) < 2:
                    continue

                optimizer.zero_grad(set_to_none=True)

                b_noisy = noisy_t[batch_rows]
                b_enh = enh_t[batch_rows]
                b_anchor = anchor_stack[row_anchor_idx_t[batch_rows]]

                neg_rows = all_neg_rows[start:start + batch_size]
                neg_rows_flat = neg_rows.reshape(-1)
                neg_noisy = noisy_t[neg_rows_flat]
                neg_enh = enh_t[neg_rows_flat]

                with autocast(device_type="cuda", dtype=torch.float16, enabled=Config.USE_AMP):
                    pos_fused = model(b_noisy, b_enh)

                    # FIX: Chunk the negatives batch. 
                    # If batch_size=4096 and K=16, len(neg_noisy) is 65536.
                    # 65536 is strictly greater than the maximum CUDA grid size (65535) for grid.y/grid.z
                    # Splitting it prevents the grid launch limit exception in standard SDPA kernels.
                    if len(neg_noisy) > 32768:
                        half_idx = len(neg_noisy) // 2
                        neg_fused_1 = model(neg_noisy[:half_idx], neg_enh[:half_idx])
                        neg_fused_2 = model(neg_noisy[half_idx:], neg_enh[half_idx:])
                        neg_fused_flat = torch.cat([neg_fused_1, neg_fused_2], dim=0)
                    else:
                        neg_fused_flat = model(neg_noisy, neg_enh)

                    neg_fused = neg_fused_flat.view(len(batch_rows), K, -1)

                    d_pos = 1.0 - F.cosine_similarity(b_anchor, pos_fused, dim=-1)
                    d_negs = 1.0 - F.cosine_similarity(b_anchor.unsqueeze(1), neg_fused, dim=-1)

                    semi_hard_mask = (d_negs > d_pos.unsqueeze(1)) & (d_negs < (d_pos + margin).unsqueeze(1))
                    masked_d = torch.where(semi_hard_mask, d_negs, torch.full_like(d_negs, float("inf")))
                    has_semi_hard = semi_hard_mask.any(dim=1)
                    fallback_idx = d_negs.argmin(dim=1)
                    semi_idx = torch.where(has_semi_hard, masked_d.argmin(dim=1), fallback_idx)

                    d_neg_active = d_negs.gather(1, semi_idx.unsqueeze(1)).squeeze(1)
                    loss = F.relu(d_pos - d_neg_active + margin).mean()

                if not torch.isfinite(loss):
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
                optimizer.step()

                epoch_losses.append(loss.item())
                batch_count += 1
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

                if Config.SAVE_EVERY_N_BATCHES > 0 and batch_count % Config.SAVE_EVERY_N_BATCHES == 0:
                    torch.save({
                        "epoch": epoch,
                        "model_state": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                        "best_val_eer": best_val_eer,
                        "best_state": best_state,
                        "no_improve": no_improve,
                        "config": config_dict,
                    }, Config.CHECKPOINT_PATH + ".intermediate")

            val_result = evaluate_anchor_vs_fused(model, val_cache, device, batch_size=Config.EVAL_BATCH_SIZE)
            val_eer = val_result["eer"]
            scheduler.step(val_eer)

            print(f"\nEpoch {epoch+1:03d} | Loss: {np.mean(epoch_losses):.5f} | Val EER: {val_eer:.2f}%")
            print(f"  Genuine mean: {val_result['genuine_mean']:.4f}, Impostor mean: {val_result['impostor_mean']:.4f}")

            if val_eer < best_val_eer:
                best_val_eer = val_eer
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
                print(f"  *** NEW BEST: {best_val_eer:.2f}% ***")
            else:
                no_improve += 1

            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val_eer": best_val_eer,
                "best_state": best_state,
                "no_improve": no_improve,
                "config": config_dict,
            }, Config.CHECKPOINT_PATH)

            if no_improve >= Config.EARLY_STOPPING_PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        if best_state is not None:
            model.load_state_dict(best_state)

        test_cache = {
            "meta": meta.iloc[test_idx].copy().reset_index(drop=True),
            "noisy_emb": cache["noisy_emb"][test_idx],
            "enhanced_emb": cache["enhanced_emb"][test_idx],
            "clean_anchor": clean_anchor,
        }
        test_result = evaluate_anchor_vs_fused(model, test_cache, device, batch_size=Config.EVAL_BATCH_SIZE)

        print("\n" + "=" * 70)
        print("FINAL TEST RESULTS (anchor-vs-fused)")
        print("=" * 70)
        print(f"EER: {test_result['eer']:.2f}%")
        print(f"Genuine mean: {test_result['genuine_mean']:.4f}")
        print(f"Impostor mean: {test_result['impostor_mean']:.4f}")

        torch.save({
            "gate_state": model.state_dict(),
            "model_type": "self_attention",
            "embedding_dim": Config.EMBEDDING_DIM,
            "num_heads": Config.NUM_HEADS,
            "test_results": test_result,
        }, Config.FINAL_CHECKPOINT_PATH)
        print(f"\nSaved final checkpoint to {Config.FINAL_CHECKPOINT_PATH}")

    if __name__ == "__main__":
        train()

self()

In [ ]:
def cross():
    #!/usr/bin/env python3
    """
    Train Cross‑Attention (Q=noisy, K,V=enhanced) with triplet loss on extended cache.
    Fixed: NaN guard, lower LR, safe evaluation.
    """

    import os
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.amp import autocast
    from collections import defaultdict
    from tqdm import tqdm

    # Enable TF32 for speed
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # ----------------------------------------------------------------------
    # CONFIG
    # ----------------------------------------------------------------------
    class Config:
        CACHE_PATH = "./cache_deleaked.pt"
        CHECKPOINT_DIR = "./checkpoints"
        CHECKPOINT_PATH = "./checkpoints/cross_attention.pt"
        FINAL_CHECKPOINT_PATH = "./checkpoints/cross_attention_final.pt"

        EMBEDDING_DIM = 192
        NUM_HEADS = 4
        DROPOUT = 0.1

        BATCH_SIZE = 4096
        EVAL_BATCH_SIZE = 16384
        EPOCHS = 150
        LR = 5e-4                    # Lowered
        WEIGHT_DECAY = 1e-4
        GRAD_CLIP = 0.5              # Lowered
        USE_AMP = True
        COMPILE_MODEL = False        # Disabled to avoid attention bugs
        SAVE_EVERY_N_BATCHES = 1000

        TRIPLET_MARGIN = 0.25
        K_NEGATIVES = 16

        SCHEDULER_FACTOR = 0.5
        SCHEDULER_PATIENCE = 5
        EARLY_STOPPING_PATIENCE = 35

        EVAL_SEED = 123
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)


    # ----------------------------------------------------------------------
    # MODEL – Cross-Attention
    # ----------------------------------------------------------------------
    class CrossAttentionFusion(nn.Module):
        def __init__(self, embedding_dim=192, num_heads=4, dropout=0.1):
            super().__init__()
            assert embedding_dim % num_heads == 0
            self.num_heads = num_heads
            self.head_dim = embedding_dim // num_heads

            self.q_proj = nn.Linear(embedding_dim, embedding_dim)
            self.k_proj = nn.Linear(embedding_dim, embedding_dim)
            self.v_proj = nn.Linear(embedding_dim, embedding_dim)
            self.out_proj = nn.Linear(embedding_dim, embedding_dim)

            self.norm1 = nn.LayerNorm(embedding_dim)
            self.norm2 = nn.LayerNorm(embedding_dim)

            self.mlp = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(embedding_dim * 2, embedding_dim),
                nn.Dropout(dropout),
            )

            self.fc_out = nn.Linear(embedding_dim, embedding_dim)

        def forward(self, noisy_emb, enhanced_emb):
            B, D = noisy_emb.shape
            Q = self.q_proj(noisy_emb)
            K = self.k_proj(enhanced_emb)
            V = self.v_proj(enhanced_emb)

            Q = Q.view(B, self.num_heads, self.head_dim)
            K = K.view(B, self.num_heads, self.head_dim)
            V = V.view(B, self.num_heads, self.head_dim)

            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
            attn_out = attn_out.contiguous().view(B, D)
            attn_out = self.out_proj(attn_out)

            x = self.norm1(noisy_emb + attn_out)
            mlp_out = self.mlp(x)
            out = self.norm2(x + mlp_out)
            fused = self.fc_out(out)
            return F.normalize(fused, p=2, dim=-1)


    # ----------------------------------------------------------------------
    # EVALUATION (safe)
    # ----------------------------------------------------------------------
    def compute_eer(gen_scores, imp_scores):
        if len(gen_scores) == 0 or len(imp_scores) == 0:
            return 50.0
        scores = np.concatenate([gen_scores, imp_scores])
        labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
        order = np.argsort(scores)
        scores, labels = scores[order], labels[order]
        n_gen = np.sum(labels == 1)
        n_imp = np.sum(labels == 0)
        if n_gen == 0 or n_imp == 0:
            return 50.0
        fnr = np.cumsum(labels == 1) / n_gen
        fpr = 1.0 - np.cumsum(labels == 0) / n_imp
        idx = np.argmin(np.abs(fnr - fpr))
        return (fnr[idx] + fpr[idx]) / 2.0 * 100.0


    @torch.no_grad()
    def evaluate_anchor_vs_fused(model, cache, device, batch_size=16384):
        model.eval()
        meta = cache["meta"].reset_index(drop=True)
        clean_anchor = cache["clean_anchor"]
        noisy = torch.from_numpy(cache["noisy_emb"]).float().to(device)
        enhanced = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

        fused_chunks = []
        for start in range(0, len(noisy), batch_size):
            end = min(start + batch_size, len(noisy))
            with autocast(device_type="cuda", dtype=torch.float16, enabled=False):  # FP32 for safety
                fused = model(noisy[start:end], enhanced[start:end])
            fused_chunks.append(fused.float())
        fused = torch.cat(fused_chunks, dim=0)

        # Check for NaNs
        if torch.isnan(fused).any():
            print("[WARN] NaN detected in fused embeddings – returning 50% EER")
            return {"eer": 50.0, "genuine_mean": 0.0, "impostor_mean": 0.0}

        spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(spk_keys)}
        anchor_matrix = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in spk_keys
        ]).to(device)
        anchor_matrix = F.normalize(anchor_matrix, p=2, dim=1)

        sample_spk_ids = np.array([spk_to_idx.get((r["language"], r["speaker"]), -1) for _, r in meta.iterrows()])
        valid_mask = sample_spk_ids >= 0
        if valid_mask.sum() == 0:
            return {"eer": 50.0, "genuine_mean": 0.0, "impostor_mean": 0.0}

        valid_fused = fused[valid_mask]
        valid_ids = torch.from_numpy(sample_spk_ids[valid_mask]).to(device)

        gen_scores = F.cosine_similarity(valid_fused, anchor_matrix[valid_ids], dim=1).cpu().numpy()

        rng = np.random.RandomState(Config.EVAL_SEED)
        num_spks = len(spk_keys)
        shift = rng.randint(1, num_spks, size=len(valid_ids))
        imp_ids = torch.from_numpy((sample_spk_ids[valid_mask] + shift) % num_spks).to(device)
        imp_scores = F.cosine_similarity(valid_fused, anchor_matrix[imp_ids], dim=1).cpu().numpy()

        eer = compute_eer(gen_scores, imp_scores)
        return {"eer": eer, "genuine_mean": float(np.mean(gen_scores)), "impostor_mean": float(np.mean(imp_scores))}


    # ----------------------------------------------------------------------
    # LOAD CACHE (robust)
    # ----------------------------------------------------------------------
    def load_cache_safe(cache_path):
        print(f"Loading cache from {cache_path}")
        import zipfile
        import pickle
        try:
            data = torch.load(cache_path, map_location="cpu", weights_only=False)
        except TypeError as e:
            if "StringDtype" in str(e):
                print("Pandas version mismatch – falling back to manual unpickle...")
                with zipfile.ZipFile(cache_path, 'r') as zf:
                    with zf.open('data.pkl') as f:
                        data = pickle.load(f)
            else:
                raise

        if "meta" in data:
            meta = data["meta"].copy() if isinstance(data["meta"], pd.DataFrame) else pd.DataFrame(data["meta"])
        elif "meta_records" in data:
            meta = pd.DataFrame(data["meta_records"])
        else:
            raise KeyError("Cache missing 'meta' or 'meta_records'.")

        noisy_emb = data["noisy_emb"].cpu().numpy() if torch.is_tensor(data["noisy_emb"]) else np.asarray(data["noisy_emb"])
        enhanced_emb = data["enhanced_emb"].cpu().numpy() if torch.is_tensor(data["enhanced_emb"]) else np.asarray(data["enhanced_emb"])

        return {
            "meta": meta.reset_index(drop=True),
            "noisy_emb": noisy_emb.astype(np.float32),
            "enhanced_emb": enhanced_emb.astype(np.float32),
            "clean_anchor": data["clean_anchor"]
        }


    # ----------------------------------------------------------------------
    # TRAIN
    # ----------------------------------------------------------------------
    def train():
        device = torch.device(Config.DEVICE)
        print(f"Device: {device}")

        cache = load_cache_safe(Config.CACHE_PATH)
        meta = cache["meta"]

        clean_anchor = cache["clean_anchor"]
        valid_spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(valid_spk_keys)}
        anchor_stack = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in valid_spk_keys
        ]).to(device, non_blocking=True)
        anchor_stack = F.normalize(anchor_stack, p=2, dim=1)

        lang_arr = meta["language"].to_numpy()
        spk_arr = meta["speaker"].to_numpy()
        row_anchor_idx = np.array([spk_to_idx.get((l, s), -1) for l, s in zip(lang_arr, spk_arr)], dtype=np.int64)
        has_anchor = row_anchor_idx >= 0
        row_anchor_idx_t = torch.from_numpy(np.maximum(row_anchor_idx, 0)).long().to(device, non_blocking=True)

        train_idx = meta.index[(meta["split"] == "train") & has_anchor].to_numpy(dtype=np.int64)
        val_idx = meta.index[(meta["split"] == "val") & has_anchor].to_numpy(dtype=np.int64)
        test_idx = meta.index[(meta["split"] == "test") & has_anchor].to_numpy(dtype=np.int64)

        print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

        val_cache = {
            "meta": meta.iloc[val_idx].copy().reset_index(drop=True),
            "noisy_emb": cache["noisy_emb"][val_idx],
            "enhanced_emb": cache["enhanced_emb"][val_idx],
            "clean_anchor": clean_anchor,
        }

        noisy_t = torch.from_numpy(cache["noisy_emb"]).float().to(device, non_blocking=True)
        enh_t = torch.from_numpy(cache["enhanced_emb"]).float().to(device, non_blocking=True)

        by_speaker = meta.groupby(["language", "speaker"]).groups
        spk_rows = {k: np.array(list(v)) for k, v in by_speaker.items()}
        lang_speakers = defaultdict(list)
        for (lang, spk) in spk_rows:
            lang_speakers[lang].append(spk)
        other_speakers_map = {}
        for lang, spks in lang_speakers.items():
            spks_arr = np.array(spks, dtype=object)
            for spk in spks:
                other_speakers_map[(lang, spk)] = spks_arr[spks_arr != spk]

        model = CrossAttentionFusion(
            embedding_dim=Config.EMBEDDING_DIM,
            num_heads=Config.NUM_HEADS,
            dropout=Config.DROPOUT,
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=Config.SCHEDULER_FACTOR, patience=Config.SCHEDULER_PATIENCE)

        start_epoch = 0
        best_val_eer = float('inf')
        best_state = None
        no_improve = 0

        if os.path.exists(Config.CHECKPOINT_PATH):
            ckpt = torch.load(Config.CHECKPOINT_PATH, map_location=device, weights_only=False)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer"])
            scheduler.load_state_dict(ckpt["scheduler"])
            start_epoch = ckpt["epoch"] + 1
            best_val_eer = ckpt["best_val_eer"]
            best_state = ckpt.get("best_state")
            no_improve = ckpt.get("no_improve", 0)
            print(f"Resuming from epoch {start_epoch}, best EER {best_val_eer:.2f}%")

        if Config.COMPILE_MODEL and hasattr(torch, 'compile'):
            print("Compiling model...")
            model = torch.compile(model, mode='reduce-overhead', dynamic=False)

        K = Config.K_NEGATIVES
        margin = Config.TRIPLET_MARGIN
        batch_size = Config.BATCH_SIZE
        config_dict = {k: getattr(Config, k) for k in dir(Config) if not k.startswith('_') and not callable(getattr(Config, k))}

        for epoch in range(start_epoch, Config.EPOCHS):
            model.train()
            rng = np.random.RandomState(42 + epoch)
            perm = rng.permutation(train_idx)

            # Pre-sample negatives
            all_neg_rows = np.zeros((len(perm), K), dtype=np.int64)
            for i, r in enumerate(perm):
                lang, spk = lang_arr[r], spk_arr[r]
                others = other_speakers_map[(lang, spk)]
                rand_spks = others[rng.randint(0, len(others), size=K)]
                for k_idx, neg_spk in enumerate(rand_spks):
                    cand = spk_rows[(lang, neg_spk)]
                    all_neg_rows[i, k_idx] = cand[rng.randint(len(cand))]

            epoch_losses = []
            batch_count = 0
            pbar = tqdm(range(0, len(perm), batch_size), desc=f"Epoch {epoch+1}/{Config.EPOCHS}")

            for start in pbar:
                batch_rows = perm[start:start + batch_size]
                if len(batch_rows) < 2:
                    continue

                optimizer.zero_grad(set_to_none=True)

                b_noisy = noisy_t[batch_rows]
                b_enh = enh_t[batch_rows]
                b_anchor = anchor_stack[row_anchor_idx_t[batch_rows]]

                # Pre-computed negatives
                neg_rows = all_neg_rows[start:start + batch_size]
                neg_rows_flat = neg_rows.reshape(-1)
                neg_noisy = noisy_t[neg_rows_flat]
                neg_enh = enh_t[neg_rows_flat]

                with autocast(device_type="cuda", dtype=torch.float16, enabled=Config.USE_AMP):
                    pos_fused = model(b_noisy, b_enh)
                    neg_fused_flat = model(neg_noisy, neg_enh)
                    neg_fused = neg_fused_flat.view(len(batch_rows), K, -1)

                    d_pos = 1.0 - F.cosine_similarity(b_anchor, pos_fused, dim=-1)
                    d_negs = 1.0 - F.cosine_similarity(b_anchor.unsqueeze(1), neg_fused, dim=-1)

                    semi_hard_mask = (d_negs > d_pos.unsqueeze(1)) & (d_negs < (d_pos + margin).unsqueeze(1))
                    masked_d = torch.where(semi_hard_mask, d_negs, torch.full_like(d_negs, float("inf")))
                    has_semi_hard = semi_hard_mask.any(dim=1)
                    fallback_idx = d_negs.argmin(dim=1)
                    semi_idx = torch.where(has_semi_hard, masked_d.argmin(dim=1), fallback_idx)

                    d_neg_active = d_negs.gather(1, semi_idx.unsqueeze(1)).squeeze(1)
                    loss = F.relu(d_pos - d_neg_active + margin).mean()

                # --- NaN guard ---
                if torch.isnan(loss) or not torch.isfinite(loss):
                    print(f"[WARN] NaN loss at batch {start}, skipping.")
                    optimizer.zero_grad(set_to_none=True)
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
                optimizer.step()

                epoch_losses.append(loss.item())
                batch_count += 1
                pbar.set_postfix({"loss": f"{loss.item():.4f}"})

                if Config.SAVE_EVERY_N_BATCHES > 0 and batch_count % Config.SAVE_EVERY_N_BATCHES == 0:
                    torch.save({
                        "epoch": epoch,
                        "model_state": model.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(),
                        "best_val_eer": best_val_eer,
                        "best_state": best_state,
                        "no_improve": no_improve,
                        "config": config_dict,
                    }, Config.CHECKPOINT_PATH + ".intermediate")

            # Validation
            val_result = evaluate_anchor_vs_fused(model, val_cache, device, batch_size=Config.EVAL_BATCH_SIZE)
            val_eer = val_result["eer"]
            scheduler.step(val_eer)

            print(f"\nEpoch {epoch+1:03d} | Loss: {np.mean(epoch_losses) if epoch_losses else float('nan'):.5f} | Val EER: {val_eer:.2f}%")
            print(f"  Genuine mean: {val_result['genuine_mean']:.4f}, Impostor mean: {val_result['impostor_mean']:.4f}")

            if val_eer < best_val_eer:
                best_val_eer = val_eer
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
                print(f"  *** NEW BEST: {best_val_eer:.2f}% ***")
            else:
                no_improve += 1

            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val_eer": best_val_eer,
                "best_state": best_state,
                "no_improve": no_improve,
                "config": config_dict,
            }, Config.CHECKPOINT_PATH)

            if no_improve >= Config.EARLY_STOPPING_PATIENCE:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        # Restore best
        if best_state is not None:
            model.load_state_dict(best_state)

        # Final test
        test_cache = {
            "meta": meta.iloc[test_idx].copy().reset_index(drop=True),
            "noisy_emb": cache["noisy_emb"][test_idx],
            "enhanced_emb": cache["enhanced_emb"][test_idx],
            "clean_anchor": clean_anchor,
        }
        test_result = evaluate_anchor_vs_fused(model, test_cache, device, batch_size=Config.EVAL_BATCH_SIZE)

        print("\n" + "=" * 70)
        print("FINAL TEST RESULTS (anchor‑vs‑fused)")
        print("=" * 70)
        print(f"EER: {test_result['eer']:.2f}%")
        print(f"Genuine mean: {test_result['genuine_mean']:.4f}")
        print(f"Impostor mean: {test_result['impostor_mean']:.4f}")

        torch.save({
            "gate_state": model.state_dict(),
            "model_type": "cross_attention",
            "embedding_dim": Config.EMBEDDING_DIM,
            "num_heads": Config.NUM_HEADS,
            "test_results": test_result,
        }, Config.FINAL_CHECKPOINT_PATH)
        print(f"\nSaved final checkpoint to {Config.FINAL_CHECKPOINT_PATH}")


    if __name__ == "__main__":
        train()

cross()

In [ ]:
def get_table():
    #!/usr/bin/env python3
    """
    Evaluate all three fusion models (MLP, Self‑Attention, Cross‑Attention)
    on the extended cache (including clean and positive SNRs).
    Generates a table with columns:
      noise_type, snr, 
      noisy_eer, noisy_minDCF, noisy_auc,
      enhanced_eer, enhanced_minDCF, enhanced_auc,
      mlp_eer, mlp_minDCF, mlp_auc,
      self_eer, self_minDCF, self_auc,
      cross_eer, cross_minDCF, cross_auc
    """

    import os
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from collections import defaultdict
    from tqdm import tqdm
    from torch.amp import autocast
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import roc_auc_score

    # ----------------------------------------------------------------------
    # 1. Model definitions (unchanged)
    # ----------------------------------------------------------------------
    class MLP(nn.Module):
        def __init__(self, embedding_dim=192, dropout=0.15):
            super().__init__()
            self.fc1 = nn.Linear(2 * embedding_dim, embedding_dim)
            self.fc2 = nn.Linear(embedding_dim, embedding_dim)
            self.fc3 = nn.Linear(embedding_dim, embedding_dim)
            self.relu = nn.ReLU()
            self.dropout = nn.Dropout(dropout)

        def forward(self, noisy_emb, enhanced_emb):
            x = torch.cat([noisy_emb, enhanced_emb], dim=-1)
            x = self.dropout(self.relu(self.fc1(x)))
            x = self.dropout(self.relu(self.fc2(x)))
            fused = self.fc3(x)
            return F.normalize(fused, p=2, dim=-1)

    class SelfAttentionFusion(nn.Module):
        def __init__(self, embedding_dim=192, num_heads=4, dropout=0.1):
            super().__init__()
            assert embedding_dim % num_heads == 0
            self.proj = nn.Linear(embedding_dim, embedding_dim)
            self.attn = nn.MultiheadAttention(
                embed_dim=embedding_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
            self.norm1 = nn.LayerNorm(embedding_dim)
            self.norm2 = nn.LayerNorm(embedding_dim)
            self.mlp = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(embedding_dim * 2, embedding_dim),
                nn.Dropout(dropout),
            )
            self.out_proj = nn.Linear(embedding_dim, embedding_dim)

        def forward(self, noisy_emb, enhanced_emb):
            n = self.proj(noisy_emb)
            e = self.proj(enhanced_emb)
            x = torch.stack([n, e], dim=1)
            attn_out, _ = self.attn(x, x, x, need_weights=False)
            x = self.norm1(x + attn_out)
            pooled = x.mean(dim=1)
            mlp_out = self.mlp(pooled)
            out = self.norm2(pooled + mlp_out)
            fused = self.out_proj(out)
            return F.normalize(fused, p=2, dim=-1)

    class CrossAttentionFusion(nn.Module):
        def __init__(self, embedding_dim=192, num_heads=4, dropout=0.1):
            super().__init__()
            assert embedding_dim % num_heads == 0
            self.num_heads = num_heads
            self.head_dim = embedding_dim // num_heads
            self.q_proj = nn.Linear(embedding_dim, embedding_dim)
            self.k_proj = nn.Linear(embedding_dim, embedding_dim)
            self.v_proj = nn.Linear(embedding_dim, embedding_dim)
            self.out_proj = nn.Linear(embedding_dim, embedding_dim)
            self.norm1 = nn.LayerNorm(embedding_dim)
            self.norm2 = nn.LayerNorm(embedding_dim)
            self.mlp = nn.Sequential(
                nn.Linear(embedding_dim, embedding_dim * 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(embedding_dim * 2, embedding_dim),
                nn.Dropout(dropout),
            )
            self.fc_out = nn.Linear(embedding_dim, embedding_dim)

        def forward(self, noisy_emb, enhanced_emb):
            B, D = noisy_emb.shape
            Q = self.q_proj(noisy_emb)
            K = self.k_proj(enhanced_emb)
            V = self.v_proj(enhanced_emb)
            Q = Q.view(B, self.num_heads, self.head_dim)
            K = K.view(B, self.num_heads, self.head_dim)
            V = V.view(B, self.num_heads, self.head_dim)
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (self.head_dim ** 0.5)
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
            attn_out = attn_out.contiguous().view(B, D)
            attn_out = self.out_proj(attn_out)
            x = self.norm1(noisy_emb + attn_out)
            mlp_out = self.mlp(x)
            out = self.norm2(x + mlp_out)
            fused = self.fc_out(out)
            return F.normalize(fused, p=2, dim=-1)

    # ----------------------------------------------------------------------
    # 2. Cache loader (unchanged)
    # ----------------------------------------------------------------------
    def load_cache_safe(cache_path):
        print(f"Loading cache from {cache_path}")
        import zipfile
        import pickle
        try:
            data = torch.load(cache_path, map_location="cpu", weights_only=False)
        except TypeError as e:
            if "StringDtype" in str(e):
                print("Pandas version mismatch – falling back to manual unpickle...")
                with zipfile.ZipFile(cache_path, 'r') as zf:
                    with zf.open('data.pkl') as f:
                        data = pickle.load(f)
            else:
                raise

        if "meta" in data:
            meta_data = data["meta"]
            if isinstance(meta_data, pd.DataFrame):
                meta = meta_data.copy()
            elif isinstance(meta_data, list):
                meta = pd.DataFrame(meta_data)
            else:
                raise TypeError(f"Unexpected meta type: {type(meta_data)}")
        elif "meta_records" in data:
            meta = pd.DataFrame(data["meta_records"])
        else:
            raise KeyError("Cache missing 'meta' or 'meta_records'.")

        noisy_emb = data["noisy_emb"].cpu().numpy() if torch.is_tensor(data["noisy_emb"]) else np.asarray(data["noisy_emb"])
        enhanced_emb = data["enhanced_emb"].cpu().numpy() if torch.is_tensor(data["enhanced_emb"]) else np.asarray(data["enhanced_emb"])

        return {
            "meta": meta.reset_index(drop=True),
            "noisy_emb": noisy_emb.astype(np.float32),
            "enhanced_emb": enhanced_emb.astype(np.float32),
            "clean_anchor": data["clean_anchor"]
        }

    # ----------------------------------------------------------------------
    # 3. Model loader (unchanged)
    # ----------------------------------------------------------------------
    def load_model(checkpoint_path, model_class, device, **kwargs):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        if "gate_state" in ckpt:
            state = ckpt["gate_state"]
        elif "model_state_dict" in ckpt:
            state = ckpt["model_state_dict"]
        else:
            state = ckpt
        if any(k.startswith("_orig_mod.") for k in state.keys()):
            state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
        model = model_class(**kwargs)
        model.load_state_dict(state)
        model.eval()
        model.to(device)
        return model

    # ----------------------------------------------------------------------
    # 4. Evaluation helpers with additional metrics
    # ----------------------------------------------------------------------
    def compute_eer(gen_scores, imp_scores):
        if len(gen_scores) == 0 or len(imp_scores) == 0:
            return 50.0
        scores = np.concatenate([gen_scores, imp_scores])
        labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
        order = np.argsort(scores)
        scores, labels = scores[order], labels[order]
        n_gen = np.sum(labels == 1)
        n_imp = np.sum(labels == 0)
        if n_gen == 0 or n_imp == 0:
            return 50.0
        fnr = np.cumsum(labels == 1) / n_gen
        fpr = 1.0 - np.cumsum(labels == 0) / n_imp
        idx = np.argmin(np.abs(fnr - fpr))
        return (fnr[idx] + fpr[idx]) / 2.0 * 100.0

    def compute_auc(gen_scores, imp_scores):
        if len(gen_scores) == 0 or len(imp_scores) == 0:
            return 0.5
        scores = np.concatenate([gen_scores, imp_scores])
        labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
        return roc_auc_score(labels, scores)

    def compute_min_dcf(gen_scores, imp_scores, prior=0.01, c_miss=1, c_fa=1):
        """
        Compute minimum Detection Cost Function (minDCF) for speaker verification.
        prior: target prior (e.g., 0.01)
        c_miss: cost of miss
        c_fa: cost of false alarm
        """
        if len(gen_scores) == 0 or len(imp_scores) == 0:
            return 1.0  # worst case
        scores = np.concatenate([gen_scores, imp_scores])
        labels = np.concatenate([np.ones(len(gen_scores)), np.zeros(len(imp_scores))])
        # sort scores descending
        order = np.argsort(scores)[::-1]
        scores = scores[order]
        labels = labels[order]

        # compute P_miss and P_fa for each threshold
        n_target = np.sum(labels == 1)
        n_non_target = np.sum(labels == 0)

        # cumulative counts
        cum_target = np.cumsum(labels == 1)
        cum_non_target = np.cumsum(labels == 0)

        # P_miss = (n_target - cum_target) / n_target
        # P_fa = cum_non_target / n_non_target
        # We'll evaluate all thresholds, including -inf and +inf
        # We can just evaluate at all score points
        P_miss = (n_target - cum_target) / n_target if n_target > 0 else 1.0
        P_fa = cum_non_target / n_non_target if n_non_target > 0 else 1.0

        # Also add one point for threshold > max (all rejected: P_miss=1, P_fa=0) and < min (all accepted: P_miss=0, P_fa=1)
        P_miss = np.concatenate([[1.0], P_miss, [0.0]])   # threshold below min (accept all) -> P_miss=0, P_fa=1
        P_fa = np.concatenate([[0.0], P_fa, [1.0]])       # threshold above max (reject all) -> P_miss=1, P_fa=0

        # Compute DCF for each point
        dcf = c_miss * P_miss * prior + c_fa * P_fa * (1 - prior)
        min_dcf = np.min(dcf)
        # Normalize by the minimum achievable DCF (if both miss and fa = 0 => 0, but we can normalize by prior or 1)
        # Usually minDCF is normalized by min(c_miss*prior, c_fa*(1-prior))
        norm = min(c_miss * prior, c_fa * (1 - prior))
        if norm > 0:
            min_dcf /= norm
        return min_dcf

    def compute_metrics(gen_scores, imp_scores):
        """Return dict with EER (%), AUC, minDCF."""
        eer = compute_eer(gen_scores, imp_scores)
        auc = compute_auc(gen_scores, imp_scores)
        min_dcf = compute_min_dcf(gen_scores, imp_scores)
        return {"eer": eer, "auc": auc, "min_dcf": min_dcf}

    @torch.no_grad()
    def compute_baseline_metrics(embeddings, cache, device, batch_size=16384):
        """Compute metrics per condition for raw embeddings (noisy or enhanced)."""
        meta = cache["meta"].reset_index(drop=True)
        clean_anchor = cache["clean_anchor"]
        emb = torch.from_numpy(embeddings).float().to(device)

        spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(spk_keys)}
        anchor_matrix = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in spk_keys
        ]).to(device)
        anchor_matrix = F.normalize(anchor_matrix, p=2, dim=1)

        sample_spk_ids = np.array([
            spk_to_idx.get((r["language"], r["speaker"]), -1) for _, r in meta.iterrows()
        ], dtype=np.int64)
        valid_mask = sample_spk_ids >= 0
        if valid_mask.sum() == 0:
            return {}, {}

        valid_ids = torch.from_numpy(sample_spk_ids[valid_mask]).to(device)
        emb_valid = emb[valid_mask]

        def metrics_for_mask(mask):
            if mask.sum() == 0:
                return {"eer": 50.0, "auc": 0.5, "min_dcf": 1.0}
            emb_sub = emb_valid[mask]
            ids_sub = valid_ids[mask]
            gen_scores = F.cosine_similarity(emb_sub, anchor_matrix[ids_sub], dim=1).cpu().numpy()
            rng = np.random.RandomState(42)
            num_spks = len(spk_keys)
            shift = rng.randint(1, num_spks, size=len(ids_sub))
            imp_ids = (ids_sub.cpu().numpy() + shift) % num_spks
            imp_scores = F.cosine_similarity(emb_sub, anchor_matrix[torch.from_numpy(imp_ids).to(device)], dim=1).cpu().numpy()
            return compute_metrics(gen_scores, imp_scores)

        cond_metrics = {}
        if "noise_type" in meta.columns and "snr" in meta.columns:
            meta_cond = meta[valid_mask].copy()
            meta_cond["snr_str"] = meta_cond["snr"].apply(lambda x: "clean" if pd.isna(x) else str(int(x)))
            groups = meta_cond.groupby(["noise_type", "snr_str"]).groups
            for (nt, snr_str), idxs in groups.items():
                mask = torch.zeros(len(emb_valid), dtype=torch.bool).to(device)
                mask[list(idxs)] = True
                cond_metrics[(nt, snr_str)] = metrics_for_mask(mask)
        overall_metrics = metrics_for_mask(torch.ones(len(emb_valid), dtype=torch.bool).to(device))
        return overall_metrics, cond_metrics

    @torch.no_grad()
    def evaluate_model_metrics(model, cache, device, batch_size=16384):
        """Returns dict with per-condition metrics for a fusion model."""
        meta = cache["meta"].reset_index(drop=True)
        clean_anchor = cache["clean_anchor"]
        noisy = torch.from_numpy(cache["noisy_emb"]).float().to(device)
        enhanced = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

        spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(spk_keys)}
        anchor_matrix = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in spk_keys
        ]).to(device)
        anchor_matrix = F.normalize(anchor_matrix, p=2, dim=1)

        sample_spk_ids = np.array([
            spk_to_idx.get((r["language"], r["speaker"]), -1) for _, r in meta.iterrows()
        ], dtype=np.int64)
        valid_mask = sample_spk_ids >= 0
        if valid_mask.sum() == 0:
            return {}

        valid_ids = torch.from_numpy(sample_spk_ids[valid_mask]).to(device)

        # Compute fused embeddings (batched)
        fused_chunks = []
        for start in range(0, len(noisy), batch_size):
            end = min(start + batch_size, len(noisy))
            with autocast(device_type="cuda", dtype=torch.float16, enabled=False):
                fused = model(noisy[start:end], enhanced[start:end])
            fused_chunks.append(fused.float())
        fused = torch.cat(fused_chunks, dim=0)

        if torch.isnan(fused).any():
            print("[WARN] NaN detected in fused embeddings – returning 50% for all conditions")
            return {}

        def metrics_for_mask(mask):
            if mask.sum() == 0:
                return {"eer": 50.0, "auc": 0.5, "min_dcf": 1.0}
            emb_sub = fused[mask]
            ids_sub = valid_ids[mask]
            gen_scores = F.cosine_similarity(emb_sub, anchor_matrix[ids_sub], dim=1).cpu().numpy()
            rng = np.random.RandomState(42)
            num_spks = len(spk_keys)
            shift = rng.randint(1, num_spks, size=len(ids_sub))
            imp_ids = (ids_sub.cpu().numpy() + shift) % num_spks
            imp_scores = F.cosine_similarity(emb_sub, anchor_matrix[torch.from_numpy(imp_ids).to(device)], dim=1).cpu().numpy()
            return compute_metrics(gen_scores, imp_scores)

        results = {}
        if "noise_type" in meta.columns and "snr" in meta.columns:
            meta_cond = meta[valid_mask].copy()
            meta_cond["snr_str"] = meta_cond["snr"].apply(lambda x: "clean" if pd.isna(x) else str(int(x)))
            groups = meta_cond.groupby(["noise_type", "snr_str"]).groups
            for (nt, snr_str), idxs in groups.items():
                mask = torch.zeros(len(fused), dtype=torch.bool).to(device)
                mask[list(idxs)] = True
                results[(nt, snr_str)] = metrics_for_mask(mask)

        # overall
        results["overall"] = metrics_for_mask(torch.ones(len(fused), dtype=torch.bool).to(device))
        return results

    # ----------------------------------------------------------------------
    # 5. Score collection for plotting (unchanged but uses new compute_eer)
    # ----------------------------------------------------------------------
    @torch.no_grad()
    def collect_scores_for_model(model, cache, device, batch_size=16384):
        meta = cache["meta"].reset_index(drop=True)
        clean_anchor = cache["clean_anchor"]
        noisy = torch.from_numpy(cache["noisy_emb"]).float().to(device)
        enhanced = torch.from_numpy(cache["enhanced_emb"]).float().to(device)

        spk_keys = list(clean_anchor.keys())
        spk_to_idx = {k: i for i, k in enumerate(spk_keys)}
        anchor_matrix = torch.stack([
            torch.as_tensor(clean_anchor[k], dtype=torch.float32) for k in spk_keys
        ]).to(device)
        anchor_matrix = F.normalize(anchor_matrix, p=2, dim=1)

        sample_spk_ids = np.array([
            spk_to_idx.get((r["language"], r["speaker"]), -1) for _, r in meta.iterrows()
        ], dtype=np.int64)
        valid_mask = sample_spk_ids >= 0
        valid_ids = torch.from_numpy(sample_spk_ids[valid_mask]).to(device)

        fused_chunks = []
        for start in range(0, len(noisy), batch_size):
            end = min(start + batch_size, len(noisy))
            fused = model(noisy[start:end], enhanced[start:end])
            fused_chunks.append(fused.float())
        fused = torch.cat(fused_chunks, dim=0)

        fused = fused[valid_mask]
        ids = valid_ids

        gen_scores = F.cosine_similarity(fused, anchor_matrix[ids], dim=1).cpu().numpy()
        rng = np.random.RandomState(42)
        num_spks = len(spk_keys)
        shift = rng.randint(1, num_spks, size=len(ids))
        imp_ids = (ids.cpu().numpy() + shift) % num_spks
        imp_scores = F.cosine_similarity(fused, anchor_matrix[torch.from_numpy(imp_ids).to(device)], dim=1).cpu().numpy()

        return gen_scores, imp_scores

    def plot_score_distributions(model_names, models, cache, device):
        plt.figure(figsize=(15, 5))
        for i, (name, model) in enumerate(zip(model_names, models)):
            gen, imp = collect_scores_for_model(model, cache, device)
            plt.subplot(1, 3, i+1)
            sns.histplot(gen, bins=50, color='green', alpha=0.5, label='Genuine', stat='density')
            sns.histplot(imp, bins=50, color='red', alpha=0.5, label='Impostor', stat='density')
            plt.xlabel('Cosine Similarity')
            plt.ylabel('Density')
            metrics = compute_metrics(gen, imp)
            plt.title(f'{name}\nEER={metrics["eer"]:.2f}%, AUC={metrics["auc"]:.3f}, minDCF={metrics["min_dcf"]:.3f}')
            plt.legend()
        plt.tight_layout()
        plt.savefig('score_distributions.png')
        plt.show()

    # ----------------------------------------------------------------------
    # 6. Main evaluation
    # ----------------------------------------------------------------------
    def main():
        CACHE_PATH = "./cache_deleaked.pt"
        MLP_CKPT = "./checkpoints/mlp_final.pt"
        SELF_CKPT = "./checkpoints/self_attention_final.pt"
        CROSS_CKPT = "./checkpoints/cross_attention_final.pt"
        DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
        BATCH_SIZE = 16384
        OUTPUT_CSV = "./evaluation_table_full.csv"

        device = torch.device(DEVICE)
        print(f"Using device: {device}")

        cache = load_cache_safe(CACHE_PATH)
        meta = cache["meta"]
        test_mask = meta["split"] == "test"
        test_cache = {
            "meta": meta[test_mask].reset_index(drop=True),
            "noisy_emb": cache["noisy_emb"][test_mask],
            "enhanced_emb": cache["enhanced_emb"][test_mask],
            "clean_anchor": cache["clean_anchor"],
        }
        print(f"Test set size: {len(test_cache['meta'])}")

        # ---- Baselines ----
        print("\nEvaluating noisy baseline...")
        noisy_overall, noisy_conds = compute_baseline_metrics(test_cache["noisy_emb"], test_cache, device, BATCH_SIZE)
        print("Evaluating enhanced baseline...")
        enh_overall, enh_conds = compute_baseline_metrics(test_cache["enhanced_emb"], test_cache, device, BATCH_SIZE)

        # ---- Models ----
        model_names = ["MLP", "SelfAttention", "CrossAttention"]
        model_configs = [
            ("MLP", MLP_CKPT, MLP, {"embedding_dim": 192, "dropout": 0.15}),
            ("SelfAttention", SELF_CKPT, SelfAttentionFusion, {"embedding_dim": 192, "num_heads": 4, "dropout": 0.1}),
            ("CrossAttention", CROSS_CKPT, CrossAttentionFusion, {"embedding_dim": 192, "num_heads": 4, "dropout": 0.1}),
        ]

        loaded_models = []
        results = {}
        for name, ckpt_path, model_cls, kwargs in model_configs:
            print(f"\nLoading {name} from {ckpt_path}")
            if not os.path.exists(ckpt_path):
                print(f"Checkpoint not found: {ckpt_path}")
                results[name] = None
                loaded_models.append(None)
                continue
            model = load_model(ckpt_path, model_cls, device, **kwargs)
            loaded_models.append(model)
            res = evaluate_model_metrics(model, test_cache, device, BATCH_SIZE)
            results[name] = res

        # ---- Build the table ----
        all_conds = set(noisy_conds.keys()) | set(enh_conds.keys())
        for res in results.values():
            if res is not None:
                all_conds.update(res.keys())
        all_conds = {k for k in all_conds if k != "overall"}
        all_conds = sorted(all_conds, key=lambda x: (x[0], x[1] if x[1] != "clean" else "zzz"))

        rows = []
        for nt, snr_str in all_conds:
            row = {
                "noise_type": nt,
                "snr": snr_str,
                "noisy_eer": noisy_conds.get((nt, snr_str), {}).get("eer", 50.0),
                "noisy_minDCF": noisy_conds.get((nt, snr_str), {}).get("min_dcf", 1.0),
                "noisy_auc": noisy_conds.get((nt, snr_str), {}).get("auc", 0.5),
                "enhanced_eer": enh_conds.get((nt, snr_str), {}).get("eer", 50.0),
                "enhanced_minDCF": enh_conds.get((nt, snr_str), {}).get("min_dcf", 1.0),
                "enhanced_auc": enh_conds.get((nt, snr_str), {}).get("auc", 0.5),
            }
            for name in model_names:
                if results[name] is not None:
                    res = results[name].get((nt, snr_str), {})
                    row[f"{name.lower()}_eer"] = res.get("eer", 50.0)
                    row[f"{name.lower()}_minDCF"] = res.get("min_dcf", 1.0)
                    row[f"{name.lower()}_auc"] = res.get("auc", 0.5)
                else:
                    row[f"{name.lower()}_eer"] = 50.0
                    row[f"{name.lower()}_minDCF"] = 1.0
                    row[f"{name.lower()}_auc"] = 0.5
            rows.append(row)

        df = pd.DataFrame(rows)

        # Add overall average row
        avg_row = {
            "noise_type": "Average",
            "snr": "",
            "noisy_eer": df["noisy_eer"].mean(),
            "noisy_minDCF": df["noisy_minDCF"].mean(),
            "noisy_auc": df["noisy_auc"].mean(),
            "enhanced_eer": df["enhanced_eer"].mean(),
            "enhanced_minDCF": df["enhanced_minDCF"].mean(),
            "enhanced_auc": df["enhanced_auc"].mean(),
        }
        for name in model_names:
            avg_row[f"{name.lower()}_eer"] = df[f"{name.lower()}_eer"].mean()
            avg_row[f"{name.lower()}_minDCF"] = df[f"{name.lower()}_minDCF"].mean()
            avg_row[f"{name.lower()}_auc"] = df[f"{name.lower()}_auc"].mean()
        df.loc[len(df)] = avg_row

        # Print table
        print("\n" + "=" * 120)
        print("EVALUATION RESULTS (EER%, minDCF, AUC)")
        print("=" * 120)
        # Show only a subset of columns for readability, but we can print all
        print(df.to_string(index=False, float_format="%.3f"))

        # Save CSV
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nResults saved to {OUTPUT_CSV}")

        # Summary of overall averages
        print("\n" + "=" * 120)
        print("OVERALL AVERAGE METRICS (across all conditions)")
        print("=" * 120)
        metrics = ["eer", "minDCF", "auc"]
        for metric in metrics:
            print(f"\n--- {metric.upper()} ---")
            print(f"  Noisy      : {df['noisy_'+metric].mean():.3f}")
            print(f"  Enhanced   : {df['enhanced_'+metric].mean():.3f}")
            for name in model_names:
                col = f"{name.lower()}_{metric}"
                if col in df.columns:
                    print(f"  {name:12s}: {df[col].mean():.3f}")

        # Best model per metric
        print("\n" + "=" * 120)
        print("BEST MODEL PER METRIC")
        print("=" * 120)
        for metric in metrics:
            cols = [f"{name.lower()}_{metric}" for name in model_names if f"{name.lower()}_{metric}" in df.columns]
            if cols:
                best_col = min(cols, key=lambda c: df[c].mean() if metric != "auc" else -df[c].mean())  # for auc, higher is better
                best_val = df[best_col].mean()
                direction = "min" if metric != "auc" else "max"
                print(f"{metric.upper():8s}: best = {best_col.replace('_'+metric,'').title()} ({direction} = {best_val:.3f})")

        # ---- Plot score distributions ----
        valid_models = [(name, model) for name, model in zip(model_names, loaded_models) if model is not None]
        if valid_models:
            plot_score_distributions(
                [name for name, _ in valid_models],
                [model for _, model in valid_models],
                test_cache,
                device
            )
        else:
            print("No valid models to plot.")

    if __name__ == "__main__":
        main()

get_table()

Using device: cuda
Loading cache from ./cache_deleaked.pt
Test set size: 136472

Evaluating noisy baseline...
Evaluating enhanced baseline...

Loading MLP from ./checkpoints/mlp_final.pt

Loading SelfAttention from ./checkpoints/self_attention_final.pt

Loading CrossAttention from ./checkpoints/cross_attention_final.pt

========================================================================================================================
EVALUATION RESULTS (EER%, minDCF, AUC)
========================================================================================================================
noise_type   snr  noisy_eer  noisy_minDCF  noisy_auc  enhanced_eer  enhanced_minDCF  enhanced_auc  mlp_eer  mlp_minDCF  mlp_auc  selfattention_eer  selfattention_minDCF  selfattention_auc  crossattention_eer  crossattention_minDCF  crossattention_auc
    babble   -10     29.975         0.915      0.779        32.376            0.943         0.746   25.421       0.968    0.823             23.451                 0.930              0.846              24.149                  0.905               0.840
    babble   -15     37.054         0.960      0.691        37.218            0.979         0.684   31.309       0.980    0.753             30.181                 0.994              0.765              30.899                  0.974               0.759
    babble   -20     43.250         0.989      0.600        41.280            0.981         0.631   36.459       0.998    0.680             36.151                 0.985              0.686              36.890                  0.983               0.681
    babble    -5     19.163         0.739      0.890        22.569            0.817         0.862   17.111       0.926    0.908             15.183                 0.867              0.924              15.490                  0.825               0.923
    babble     0      9.684         0.496      0.963        12.002            0.680         0.950    9.376       0.854    0.965              7.612                 0.649              0.976               7.796                  0.650               0.975
    babble    10      2.770         0.244      0.995         3.098            0.308         0.994    3.980       0.677    0.991              2.770                 0.427              0.995               2.626                  0.407               0.996
    babble    15      1.888         0.206      0.997         2.113            0.261         0.997    3.447       0.682    0.993              2.400                 0.433              0.997               2.154                  0.287               0.998
    babble    20      1.539         0.143      0.998         1.805            0.215         0.998    3.344       0.600    0.994              2.195                 0.341              0.998               1.929                  0.270               0.998
    babble     5      4.883         0.352      0.987         5.909            0.442         0.984    6.217       0.682    0.982              4.780                 0.518              0.990               4.534                  0.422               0.990
     clean clean      1.190         0.115      0.999         1.580            0.195         0.999    2.995       0.590    0.995              2.113                 0.331              0.998               1.847                  0.240               0.999
     music   -10     18.096         0.632      0.903        19.101            0.733         0.890   12.967       0.907    0.939             11.777                 0.813              0.951              11.572                  0.724               0.952
     music   -15     24.313         0.771      0.842        26.098            0.880         0.820   18.978       0.946    0.890             17.952                 0.818              0.901              18.055                  0.768               0.900
     music   -20     30.119         0.811      0.780        32.007            0.866         0.753   25.236       0.955    0.833             23.636                 0.893              0.849              24.313                  0.819               0.846
     music    -5     11.736         0.525      0.950        11.982            0.588         0.946    8.412       0.795    0.971              7.099                 0.601              0.980               7.243                  0.572               0.979
     music     0      6.914         0.347      0.977         6.463            0.467         0.978    5.047       0.664    0.987              3.693                 0.475              0.993               3.549                  0.367               0.994
     music    10      2.257         0.202      0.997         2.565            0.282         0.996    3.324       0.572    0.994              2.257                 0.349              0.997               2.072                  0.260               0.998
     music    15      1.785         0.157      0.998         1.929            0.251         0.998    3.160       0.612    0.994              2.216                 0.365              0.997               2.011                  0.268               0.998
     music    20      1.395         0.135      0.998         1.744            0.209         0.998    3.201       0.605    0.995              2.298                 0.334              0.998               1.888                  0.268               0.998
     music     5      3.549         0.262      0.992         3.611            0.335         0.992    3.673       0.726    0.992              2.606                 0.472              0.996               2.647                  0.441               0.996
     noise   -10     16.352         0.486      0.911        16.803            0.549         0.910    9.992       0.821    0.961              9.171                 0.624              0.970               9.376                  0.627               0.970
     noise   -15     20.866         0.622      0.878        21.666            0.715         0.866   13.254       0.941    0.941             12.126                 0.818              0.952              11.941                  0.718               0.953
     noise   -20     26.303         0.669      0.825        28.190            0.810         0.804   17.706       0.943    0.907             17.173                 0.843              0.915              17.234                  0.780               0.916
     noise    -5     11.079         0.384      0.950        10.895            0.494         0.952    6.750       0.691    0.980              5.540                 0.539              0.986               5.786                  0.493               0.986
     noise     0      7.119         0.330      0.975         6.955            0.380         0.977    4.822       0.653    0.989              3.652                 0.409              0.994               3.508                  0.436               0.994
     noise    10      2.380         0.167      0.997         2.688            0.304         0.996    3.467       0.618    0.994              2.298                 0.341              0.997               2.093                  0.309               0.998
     noise    15      1.703         0.159      0.998         2.011            0.243         0.998    3.098       0.632    0.994              2.113                 0.360              0.998               1.929                  0.246               0.998
     noise    20      1.375         0.129      0.999         1.764            0.200         0.998    3.385       0.543    0.995              2.195                 0.308              0.998               1.908                  0.256               0.998
     noise     5      3.673         0.289      0.991         3.529            0.338         0.993    4.001       0.689    0.991              2.749                 0.418              0.996               2.688                  0.459               0.996
   Average           12.229         0.437      0.924        12.855            0.517         0.918   10.362       0.760    0.944              9.192                 0.581              0.952               9.219                  0.528               0.951

Results saved to ./evaluation_table_full.csv

========================================================================================================================
OVERALL AVERAGE METRICS (across all conditions)
========================================================================================================================

--- EER ---
  Noisy      : 12.229
  Enhanced   : 12.855
  MLP         : 10.362
  SelfAttention: 9.192
  CrossAttention: 9.219

--- MINDCF ---
  Noisy      : 0.437
  Enhanced   : 0.517
  MLP         : 0.760
  SelfAttention: 0.581
  CrossAttention: 0.528

--- AUC ---
  Noisy      : 0.924
  Enhanced   : 0.918
  MLP         : 0.944
  SelfAttention: 0.952
  CrossAttention: 0.951

========================================================================================================================
BEST MODEL PER METRIC
========================================================================================================================
EER     : best = Selfattention (min = 9.192)
MINDCF  : best = Crossattention (min = 0.528)
AUC     : best = Selfattention (max = 0.952)

In [ ]:
fs.put("/marimo/checkpoints/cross_attention_final.pt", "Data_Science_Project/Models/New models/")

fs.put("/marimo/checkpoints/mlp_final.pt", "Data_Science_Project/Models/New models/")

fs.put("/marimo/checkpoints/self_attention_final.pt", "Data_Science_Project/Models/New models/")